# 01 - Document Loading

## Objective

This notebook validates the document ingestion pipeline.

Supported document types:

- PDF
- DOCX
- XLSX

Documents are loaded from the `docs` folder and converted into LangChain `Document` objects.

## Expected Outcomes

- Verify document loading
- Verify metadata generation
- Validate content extraction
- Understand the structure of loaded documents

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project Root: {project_root}")

Project Root: d:\AI\Projects\oracle-rag-comparison-lab


In [4]:
from src.config import DOCS_PATH
from src.loaders.document_loader import DocumentLoader

print(f"Documents Path: {DOCS_PATH}")

Documents Path: ../docs


## Load Documents

The custom document loader:

- Reads PDF files
- Reads DOCX files
- Reads Excel files
- Creates LangChain `Document` objects
- Captures metadata

In [5]:
loader = DocumentLoader(DOCS_PATH)

documents = loader.load()

Starting Document Loading...

Loading PDF : iSupplier_portal.pdf

Loading PDF : Purchasing_manual.pdf


DOCUMENT LOADING SUMMARY
PDF Files Loaded   : 2
DOCX Files Loaded  : 0
XLSX Files Loaded  : 0
Total Documents    : 1502


## Verify Documents Were Loaded
``

In [6]:
print(f"Total Documents Loaded: {len(documents)}")

Total Documents Loaded: 1502


## Inspect First Document

In [7]:
documents[0]

Document(metadata={'source': 'iSupplier_portal.pdf', 'file_type': 'pdf', 'page': 1}, page_content="Oracle® iSupplier Portal\n User's Guide \nRelease 12.2\n Part No. E48972-13\nOctober 2025")

## View Metadata

Metadata will later be used for:

- Source Citations
- File References
- Page References
- Retrieval Diagnostics
- RAG Evaluation

In [8]:
documents[0].metadata

{'source': 'iSupplier_portal.pdf', 'file_type': 'pdf', 'page': 1}

## Preview First Document Content

Validate that text extraction from the source document has worked correctly.

In [9]:
print("=" * 100)

print("FIRST DOCUMENT PREVIEW")

print("=" * 100)

print(
    documents[0].page_content[:2000]
)

FIRST DOCUMENT PREVIEW
Oracle® iSupplier Portal
 User's Guide 
Release 12.2
 Part No. E48972-13
October 2025


## Document Source Breakdown

Understand how many LangChain documents were generated from each source file.

In [10]:
source_counts = {}

for document in documents:

    source = document.metadata.get(
        "source",
        "Unknown"
    )

    source_counts[source] = (
        source_counts.get(source, 0) + 1
    )

print("=" * 100)
print("DOCUMENT SOURCE BREAKDOWN")
print("=" * 100)

for source, count in source_counts.items():

    print(f"\n{source}")

    print(f"Documents: {count}")

DOCUMENT SOURCE BREAKDOWN

iSupplier_portal.pdf
Documents: 191

Purchasing_manual.pdf
Documents: 1311


## Document Type Breakdown

Review the number of documents generated for each document type.

In [11]:
type_counts = {}

for document in documents:

    file_type = document.metadata.get(
        "file_type",
        "unknown"
    )

    type_counts[file_type] = (
        type_counts.get(file_type, 0) + 1
    )

print("=" * 100)
print("DOCUMENT TYPE BREAKDOWN")
print("=" * 100)

for file_type, count in type_counts.items():

    print(
        f"{file_type.upper():<10} : {count}"
    )

DOCUMENT TYPE BREAKDOWN
PDF        : 1502


## Sample Documents

Review a few loaded documents and their metadata.

In [12]:
for idx, document in enumerate(documents[:3]):

    print("\n" + "=" * 100)

    print(f"SAMPLE DOCUMENT #{idx + 1}")

    print("=" * 100)

    print("Metadata")

    print(document.metadata)

    print("\nContent Preview")

    print(document.page_content[:500])


SAMPLE DOCUMENT #1
Metadata
{'source': 'iSupplier_portal.pdf', 'file_type': 'pdf', 'page': 1}

Content Preview
Oracle® iSupplier Portal
 User's Guide 
Release 12.2
 Part No. E48972-13
October 2025

SAMPLE DOCUMENT #2
Metadata
{'source': 'iSupplier_portal.pdf', 'file_type': 'pdf', 'page': 2}

Content Preview
Oracle iSupplier Portal User's Guide , Release 12.2
Part No. E48972-13
Copyright © 2009, 2025, Oracle and/or its affiliates. 
Primary Author:     Chetna Arora
Contributing Author:     Gowri Arur, Pragya Nair, Smile Nagpal
This software and related documentation are provided under a license agreement containing restrictions on 
use and disclosure and are protected by intellectual property laws. Except as expressly permitted in your 
license agreement or allowed by law, you may not use, copy, repr

SAMPLE DOCUMENT #3
Metadata
{'source': 'iSupplier_portal.pdf', 'file_type': 'pdf', 'page': 3}

Content Preview
Access to Oracle Support
Oracle customer access to and use of Oracle support 

## Document Length Statistics

Understand the size distribution of ingested documents before chunking.

In [13]:
lengths = [
    len(document.page_content)
    for document in documents
]

print("=" * 100)

print("DOCUMENT LENGTH STATISTICS")

print("=" * 100)

print(f"Minimum Length : {min(lengths):,}")

print(f"Maximum Length : {max(lengths):,}")

print(
    f"Average Length : {sum(lengths)/len(lengths):,.0f}"
)

DOCUMENT LENGTH STATISTICS
Minimum Length : 2
Maximum Length : 4,370
Average Length : 1,658


## Validation Checklist

Verify the following before proceeding to Notebook 02.

Expected Outcomes:

- Documents Loaded Successfully
- Metadata Available
- PDF Content Extracted
- DOCX Content Extracted
- XLSX Content Extracted
- Source References Captured

In [14]:
print("=" * 100)

print("DOCUMENT LOADING VALIDATION")

print("=" * 100)

print(
    f"Documents Loaded Successfully : {'✅' if len(documents) > 0 else '❌'}"
)

print(
    f"Metadata Available            : {'✅' if documents and documents[0].metadata else '❌'}"
)

print(
    f"Content Available             : {'✅' if documents and documents[0].page_content else '❌'}"
)

DOCUMENT LOADING VALIDATION
Documents Loaded Successfully : ✅
Metadata Available            : ✅
Content Available             : ✅


# Success Criteria

If all validation checks pass, proceed to:

## Notebook 02 - Recursive Chunking

Next we will:

- Create chunks using RecursiveCharacterTextSplitter
- Analyze chunk sizes
- Compare chunk distributions
- Prepare data for embeddings